# DS Week 01 — Calibrated Credit Risk Under Distribution Shift
Decision-grade probability modeling for next-month default: leakage control, domain features, calibration, uncertainty, subgroup diagnostics and shift stress tests.

## 1. Business framing and research hypothesis
Ranking alone is insufficient in credit risk because decisions depend on probability quality and asymmetric costs. We test whether behavioral features and gradient boosting improve discrimination, whether post-hoc calibration improves probability quality, and how uncertainty behaves under controlled shift.

In [ ]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
ROOT=Path.cwd(); sys.path.insert(0,str(ROOT)) if str(ROOT) not in sys.path else None
from src.synthetic import make_credit_like
from src.features import model_matrix, engineer_credit_features
from src.metrics import evaluate_probabilities, cost_optimal_threshold
from src.calibration import PlattCalibratedModel
from src.conformal import conformal_quantile,prediction_sets,coverage_and_size
SEED=42; SMOKE_MODE=True


## 2. Dataset provenance, target and leakage audit
Canonical dataset: UCI Default of Credit Card Clients. `ID` is excluded. `SEX`, `MARRIAGE`, `EDUCATION`, and `AGE` are withheld from the main model and retained only for diagnostic slices. Historical billing/payment fields precede the next-month target.

In [ ]:
def load_real():
    from ucimlrepo import fetch_ucirepo
    d=fetch_ucirepo(id=350); return d.data.features.copy(), d.data.targets.squeeze().astype(int)
X_raw,y=make_credit_like(5000,SEED) if SMOKE_MODE else load_real()
print(X_raw.shape, y.value_counts(normalize=True).round(3).to_dict())
print(pd.DataFrame({'missing':X_raw.isna().sum(),'unique':X_raw.nunique()}).sort_values('missing',ascending=False).head(10))


## 3. Domain-specific feature engineering
Create utilization level/volatility, payment ratios, bill/payment slopes, delinquency counts/severity, recency-weighted delinquency and recent-vs-old delinquency trend. All transformations are row-wise and target-free.

In [ ]:
X_model=model_matrix(X_raw)
print('model features',X_model.shape[1])
print(sorted(set(X_model.columns)-set(X_raw.columns)))


## 4. Train / calibration / test design
Use a dedicated calibration split. The benchmark has no genuine future deployment split, so later distribution-shift experiments are labeled as controlled sensitivity tests rather than temporal validation.

In [ ]:
idx=np.arange(len(y))
Xtr,Xtmp,ytr,ytmp,itr,itmp=train_test_split(X_model,y,idx,test_size=.4,stratify=y,random_state=SEED)
Xcal,Xte,ycal,yte,ical,ite=train_test_split(Xtmp,ytmp,itmp,test_size=.5,stratify=ytmp,random_state=SEED)
print(len(ytr),len(ycal),len(yte))


## 5. Competitive models and metrics
Compare a transparent Logistic Regression baseline with CPU-efficient histogram gradient boosting. Evaluate ROC-AUC, PR-AUC, Brier, log loss, ECE and recall at 5% FPR.

In [ ]:
logit=Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler()),('m',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED))])
gbdt=Pipeline([('imp',SimpleImputer(strategy='median')),('m',HistGradientBoostingClassifier(max_iter=100 if SMOKE_MODE else 180,learning_rate=.06,max_leaf_nodes=31,l2_regularization=1,min_samples_leaf=30,random_state=SEED))])
rows={}
for name,m in {'logistic':logit,'hist_gbdt':gbdt}.items():
    m.fit(Xtr,ytr); rows[name]=evaluate_probabilities(yte,m.predict_proba(Xte)[:,1])
print(pd.DataFrame(rows).T.round(4))


## 6. Probability calibration and threshold economics
Fit Platt calibration only on the held-out calibration split. Then optimize a decision threshold under an illustrative 5:1 missed-default to false-rejection cost ratio; sensitivity-test this ratio in real use.

In [ ]:
cal=PlattCalibratedModel(gbdt).fit(Xcal,ycal)
praw=gbdt.predict_proba(Xte)[:,1]; p=cal.predict_proba(Xte)[:,1]
print(pd.DataFrame({'raw':evaluate_probabilities(yte,praw),'calibrated':evaluate_probabilities(yte,p)}).T.round(4))
print('cost optimum',cost_optimal_threshold(yte,p,5,1)[:4])


## 7. Split-conformal uncertainty
Return prediction sets instead of forcing a label. Classical marginal coverage depends on exchangeability, so coverage is re-measured under stress scenarios.

In [ ]:
pcal=cal.predict_proba(Xcal)[:,1]; qhat=conformal_quantile(ycal,pcal,.1); sets=prediction_sets(p,qhat); print({'qhat':qhat,'coverage':coverage_and_size(yte,sets)})


## 8. Feature selection and interpretability
Use permutation importance on held-out data rather than in-sample impurity scores.

In [ ]:
pi=permutation_importance(gbdt,Xte,yte,scoring='average_precision',n_repeats=3,random_state=SEED,n_jobs=-1)
imp=pd.DataFrame({'feature':Xte.columns,'importance':pi.importances_mean}).sort_values('importance',ascending=False).head(15); print(imp)


## 9. Subgroup slice diagnostics
Evaluate errors/calibration across withheld demographic/proxy slices. Slice differences are diagnostics, not legal fairness conclusions.

In [ ]:
raw_test=X_raw.iloc[ite].reset_index(drop=True); yy=yte.to_numpy(); out=[]
for col in ['SEX','EDUCATION','MARRIAGE']:
    for v in sorted(raw_test[col].unique()):
        mask=raw_test[col].to_numpy()==v
        if mask.sum()>=30 and len(np.unique(yy[mask]))==2:
            m=evaluate_probabilities(yy[mask],p[mask]); out.append({'slice':f'{col}={v}','n':int(mask.sum()),'pr_auc':m['pr_auc'],'brier':m['brier']})
print(pd.DataFrame(out).sort_values('pr_auc').head(12))


## 10. Controlled distribution-shift stress test
Increase utilization and/or delinquency in test covariates while holding labels fixed. These are sensitivity probes, not causal counterfactuals or temporal validation.

In [ ]:
def shifted(raw,util=1.0,delinq=0):
    x=raw.copy()
    for i in range(1,7): x[f'BILL_AMT{i}']*=util
    for c in ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']: x[c]=np.clip(x[c]+delinq,-2,9)
    return model_matrix(x)
for name,x in {'base':model_matrix(raw_test),'util+20%':shifted(raw_test,1.2,0),'delinq+1':shifted(raw_test,1,1),'combined':shifted(raw_test,1.2,1)}.items():
    ps=cal.predict_proba(x)[:,1]; s=prediction_sets(ps,qhat); print(name,evaluate_probabilities(yy,ps),coverage_and_size(yy,s))


## 11. Conclusions, limitations and next research step
Use the real UCI run for portfolio claims. The smoke mode validates implementation only. Next: repeat on a genuinely temporal Home Credit stability benchmark; compare CatBoost/LightGBM/TabPFN under the same calibration protocol; evaluate recalibration triggers; investigate reject inference and selection bias; compare standard with shift-adaptive conformal methods.

**Interview focus:** why PR-AUC, why a calibration split, why calibration and ranking differ, why stress tests are not temporal validation, and where conformal guarantees can fail.